# loading and preprocessing data

In [ ]:
import torchvision 
from torchvision import transforms
image_path = './'
transform = transforms.Compose([
    transforms.ToTensor()
])
mnist_dataset = torchvision.datasets.MNIST(
    root=image_path, train=True,
    transform=transform, download=True
)


ModuleNotFoundError: No module named 'torchvision'

In [ ]:
from torch.utils.data import Subset
import torch
mnist_valid_dataset = Subset(mnist_dataset, torch.arange(10000))
mnist_train_dataset = Subset(mnist_dataset, torch.arange(10000, len(mnist_dataset)))


In [ ]:
mnist_test_dataset = torchvision.datasets.MNIST(
    root=image_path, train=False,
    transform=transform, download=False
)

from torch.utils.data import DataLoader
batch_size = 64
torch.manual_seed(1)
train_dl = DataLoader(mnist_train_dataset, batch_size, shuffle=True)
valid_dl = DataLoader(mnist_valid_dataset, batch_size, shuffle=False)


# Configuring CNN layers in PyTorch
1. Conv2d Layer
python
nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, padding=2)
in_channels: number of input channels (MNIST = 1 for grayscale).

out_channels: number of filters (32 feature maps).

kernel_size=5: each filter is a 5×5 window scanning the image.

padding=2: adds zeros around the input so output size stays the same.

3. Data Format (NCHW)
PyTorch expects tensors in NCHW format:

N = batch size (number of images).

C = channels (grayscale=1, RGB=3).

H = height.

W = width.

Pooling Layers
python
nn.MaxPool2d(kernel_size=2, stride=2)
MaxPool2d: takes the maximum value in each 2×2 block.

AvgPool2d: takes the average instead.

stride=2: halves the spatial dimensions.

Conv2d: Extracts local features (edges, curves).

Pooling: Shrinks feature maps, keeps important signals.

Dropout: Adds regularization during training.

Fully Connected Layers: Combine features into final classification.

# constructing cnn using pytorch


In [ ]:
from torch import nn
model = nn.Sequential()
#1st convulation layer
model.add_module('conv1', nn.Conv2d(in_channels= 1, out_channels= 34, kernel_size= 5, padding = 2))

#activation function
model.add_module('relu1', nn.ReLU())

#1st pooling 
model.add_module('pool1', nn.MaxPool2d(kernel_size= 2))

#2n convulation layer
model.add_module('conv2', nn.Conv2d(in_channels= 34, out_channels= 64, kernel_size= 5))


model.add_module('relu2', nn.ReLU())
model.add_module('pool2', nn.MaxPool2d(kernel_size=2))


#flattening
model.add_module('flatten', nn.Flatten())
x = torch.ones((4,1,28,28))
model(x).shape

#connected layers the first one
model.add_module('fc1', nn.Linear(3136, 1024))
model.add_module('relu3', nn.ReLU())

model.add_module('dropout', nn.Dropout(p=0.5))

# finally fully connected layer
model.add_module("fc2", nn.Linear(1024, 10)) # produce 10 digits

#loss function and optimizer
loss_fn = nn.CrossEntropyLoss() # compare actual and predicted
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)


In [ ]:
def train(model, num_epochs, train_dl, valid_dl):
  loss_hist_train = [0] * num_epochs
  accuracy_hist_train = [0] * num_epochs
  loss_hist_valid = [0] * num_epochs
  accuracy_hist_valid = [0] * num_epochs

  for epoch in range(num_epochs):
    model.train()
    for x_batch, y_batch in train_dl:
      pred = model(x_batch)
      loss = loss_fn(pred, y_batch)
      loss.backward() # writes by computing weights
      optimizer.step() #copies
      optimizer.zero_grad() # erases and weight for another epoch
      loss_hist_train[epoch] += loss.item() * y_batch.size(0) #.item gets the address reaference
      is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
      accuracy_hist_train[epoch] += is_correct.sum()
      accuracy_hist_train[epoch] += is_correct.sum()
      loss_hist_train[epoch] /= len(train_dl.dataset)
      accuracy_hist_train[epoch] /= len(train_dl.dataset)

    model.eval()
    #note this is forward  where it has to record ths why we dont use the .zero_grad()
    with torch.no_grad():
        for x_batch, y_batch in valid_dl:
            pred = model(x_batch)
            loss = loss_fn(pred, y_batch)
            loss_hist_valid[epoch] += loss.item()*y_batch.size(0)

            is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
            accuracy_hist_valid[epoch] += is_correct.sum()
            loss_hist_valid[epoch] /= len(valid_dl.dataset)
            accuracy_hist_valid[epoch] /= len(valid_dl.dataset)
            
    #reporting and return
    print(f'Epoch {epoch+1} accuracy: ' f'{accuracy_hist_train[epoch]:.4f} val_accuracy: ' f'{accuracy_hist_valid[epoch]:.4f}')
    return loss_hist_train, loss_hist_valid,accuracy_hist_train, accuracy_hist_valid

            
